# DR model: hold-out evaluation (no training)

Before **Run All**: turn on **GPU T4 x2** and **Internet**, and use **Add Input** to attach
1. the dataset `ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy`
2. an uploaded dataset with `model.h5` and `model.tflite` (from `models/dr/` in the repo).

Cells 1-3 are copied unchanged from `model_notebook.ipynb` (setup, dataset path, loader) so `df` is built exactly as in training. Cell 4 rebuilds the training run's validation fold, checks it matches, scores both models and saves `holdout_predictions.csv` + `holdout_run_info.json` to the Output panel. It trains nothing.

Expected time: about 20-40 minutes.

## 1. Setup (unchanged)

In [ ]:
!pip install -q tensorflow scikit-learn seaborn --upgrade

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Dataset path (unchanged, plus a fallback for the new mount path)

In [ ]:
import kagglehub
import os

# Download latest version
DATASET_PATH = "/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy"
if not os.path.isdir(DATASET_PATH):  # new notebooks mount inputs at a shorter path
    import glob as _glob
    _hits = _glob.glob("/kaggle/input/*eyepacs-aptos-messidor*") + _glob.glob("/kaggle/input/*/*/*eyepacs-aptos-messidor*")
    assert _hits, "Dataset not attached: Add Input -> search eyepacs-aptos-messidor-diabetic-retinopathy"
    DATASET_PATH = _hits[0]
print("Path to dataset files:", DATASET_PATH)

# Inspect the folder structure (first 3 levels) so we know exactly how to load it,
# rather than assuming a structure and hitting FileNotFoundError later.
for root, dirs, files in os.walk(DATASET_PATH):
    depth = root[len(DATASET_PATH):].count(os.sep)
    if depth <= 2:
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root) or root}/  ({len(files)} files, {len(dirs)} subdirs)")
        for f in sorted(files)[:3]:
            print(f"{indent}  - {f}")


## 3. Build df (unchanged)

In [ ]:
import glob
import re
import pandas as pd

IMG_EXTS = (".png", ".jpg", ".jpeg")

csv_files = glob.glob(os.path.join(DATASET_PATH, "**", "*.csv"), recursive=True)
print("CSV files found:", csv_files)

records = []

if csv_files:
    # CSV-driven layout, e.g. an id/diagnosis column per row (same shape as APTOS's train_1.csv)
    for csv_path in csv_files:
        sub = pd.read_csv(csv_path)
        base_dir = os.path.dirname(csv_path)
        id_col = next((c for c in sub.columns if "id" in c.lower() or "image" in c.lower() or "file" in c.lower()), None)
        label_col = next((c for c in sub.columns if any(k in c.lower() for k in ("diagnosis", "label", "grade", "level"))), None)
        if id_col is None or label_col is None:
            print(f"Skipping {csv_path}: couldn't identify id/label columns ({list(sub.columns)})")
            continue
        for _, r in sub.iterrows():
            stem = str(r[id_col])
            candidates = [stem] if stem.lower().endswith(IMG_EXTS) else [stem + ext for ext in IMG_EXTS]
            fpath = None
            for cand in candidates:
                hits = glob.glob(os.path.join(base_dir, "**", cand), recursive=True)
                if hits:
                    fpath = hits[0]
                    break
            if fpath:
                records.append({"filepath": fpath, "diagnosis": int(r[label_col])})
else:
    # Folder-per-class layout, e.g. .../train/0/*.png .../train/4/*.png
    class_dirs = [d for d in glob.glob(os.path.join(DATASET_PATH, "**", "*"), recursive=True)
                  if os.path.isdir(d) and os.path.basename(d) in {"0", "1", "2", "3", "4"}]
    for cd in class_dirs:
        label = int(os.path.basename(cd))
        for fpath in glob.glob(os.path.join(cd, "*")):
            if fpath.lower().endswith(IMG_EXTS):
                records.append({"filepath": fpath, "diagnosis": label})

df = pd.DataFrame(records)
assert len(df) > 0, "No images matched -- check the printed folder structure above and adjust the loader."

df["label"] = (df["diagnosis"] > 0).astype(int)  # 0 = No DR, 1 = DR present (same binarization as before)

print(f"Total samples found: {len(df)}")
df.head()


## 4. Hold-out evaluation

In [ ]:
# ===================== HOLD-OUT EVALUATION CELL =====================
# Paste into ONE new cell at the bottom of notebooks/model_notebook.ipynb on Kaggle, AFTER running
# code cells 1 (setup + SEED), 2 (DATASET_PATH) and 3 (builds df) UNCHANGED.
# Do NOT run code cells 4-12 (visualisation, generators, training, saving).
#
# Needs two inputs added with "Add Input -> Upload -> New Dataset":
#   the dataset  ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy
#   the models   models/dr/model.h5 and models/dr/model.tflite from this repo
#                (the original names dr_screening_model.h5 / .tflite also work)
#
# What it does: rebuilds the SAME validation fold as the notebook's split cell (same code, same
# SEED), checks it matches what the notebook printed during training, predicts with BOTH the .h5
# and the .tflite model, and writes /kaggle/working/holdout_predictions.csv + holdout_run_info.json.
# If the .h5 reproduces the notebook's validation metrics, the shipped model IS the notebook's
# 8-epoch model and this fold was never trained on. If not, do not use these numbers.

import glob, json, os, re
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)      # same as the notebook's split cell
BATCH_SIZE = 32            # same as the notebook's split cell
EXPECTED = {               # taken from the saved outputs inside notebooks/model_notebook.ipynb
    "n_total": 236170,
    "n_val": 47234,
    "val_counts": {0: 27581, 1: 19653},
    "first_diagnoses": [2, 2, 2, 2, 2],
    "metrics_at_0.5": {"accuracy": 0.790, "precision": 0.920, "recall": 0.542, "f1": 0.682},
}
problems = []

# ---------- 1. Is df the same df the model was trained on? ----------
print("len(df) =", len(df), "| expected", EXPECTED["n_total"])
assert len(df) == EXPECTED["n_total"], (
    "len(df) differs from training. STOP: the dataset version is different. "
    "Do not use any numbers from this run.")
first5 = df["diagnosis"].head(5).tolist()
print("first 5 diagnoses =", first5, "| expected", EXPECTED["first_diagnoses"])
if first5 != EXPECTED["first_diagnoses"]:
    problems.append("row order of df differs from training (first 5 diagnoses)")

# ---------- 2. Rebuild the validation fold exactly as the notebook does ----------
def base_id(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r'(_aug\d*|_flip\w*|_rot(ate)?\d*|_crop\d*|_v\d+|_\d+)$', '', stem, flags=re.IGNORECASE)

df["base_id"] = df["filepath"].apply(base_id)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf.split(df, y=df["label"], groups=df["base_id"]))
val_df = df.iloc[val_idx].copy().reset_index(drop=True)
val_counts = {int(k): int(v) for k, v in val_df["label"].value_counts().items()}
print("validation rows =", len(val_df), "| class counts =", val_counts)
if len(val_df) != EXPECTED["n_val"] or val_counts != EXPECTED["val_counts"]:
    problems.append("validation fold size/class counts differ from training")

# ---------- 3. Find the model files (searches a few folder depths only) ----------
def find_file(*names):
    patterns = ["/kaggle/working/", "/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"]
    for name in names:
        for p in patterns:
            hits = glob.glob(p + name)
            if hits:
                return hits[0]
    raise FileNotFoundError(f"None of {names} found. Upload them as a Kaggle dataset (Add Input) or to /kaggle/working.")

h5_path = find_file("model.h5", "dr_screening_model.h5")
tflite_path = find_file("model.tflite", "dr_screening_model.tflite")
print("h5     :", h5_path)
print("tflite :", tflite_path)

# ---------- 4. Same preprocessing as training validation ----------
val_df["label_str"] = val_df["label"].astype(str)
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_dataframe(
    val_df, x_col="filepath", y_col="label_str",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False, seed=SEED)
assert len(val_gen.filenames) == len(val_df), "some validation images could not be read"
assert list(val_gen.filenames) == val_df["filepath"].tolist(), "generator order != dataframe order"

# ---------- 5. Predict with the .h5 model ----------
model = tf.keras.models.load_model(h5_path, compile=False)
val_gen.reset()
prob_h5 = model.predict(val_gen, verbose=1).ravel()

# ---------- 6. Predict with the .tflite model (the one the app serves) ----------
interp = tf.lite.Interpreter(model_path=tflite_path)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
assert inp["dtype"] == np.float32, f"unexpected tflite input dtype {inp['dtype']}"
prob_tfl = []
for b in range(len(val_gen)):
    xb, _ = val_gen[b]
    for img in xb:
        interp.set_tensor(inp["index"], img[None].astype(np.float32))
        interp.invoke()
        prob_tfl.append(float(interp.get_tensor(out["index"]).ravel()[0]))
prob_tfl = np.array(prob_tfl)
assert len(prob_tfl) == len(val_df)

# ---------- 7. Metrics at 0.5 and the reproducibility check ----------
y = val_df["label"].values
def report(name, p, thr=0.5):
    yhat = (p > thr).astype(int)
    m = {"accuracy": accuracy_score(y, yhat), "precision": precision_score(y, yhat),
         "recall": recall_score(y, yhat), "f1": f1_score(y, yhat)}
    print(f"\n{name} @ threshold {thr}:", {k: round(v, 3) for k, v in m.items()})
    print(confusion_matrix(y, yhat))
    return m

m_h5 = report("h5", prob_h5)
report("tflite", prob_tfl)
for k, v in EXPECTED["metrics_at_0.5"].items():
    if abs(m_h5[k] - v) > 0.002:
        problems.append(f"h5 {k} {m_h5[k]:.3f} does not match notebook value {v}")
print("\nmax |h5 - tflite| probability difference:", float(np.abs(prob_h5 - prob_tfl).max()))

# ---------- 8. Save ----------
res = pd.DataFrame({
    "filepath": val_df["filepath"], "diagnosis": val_df["diagnosis"], "label": val_df["label"],
    "prob_h5": prob_h5, "prob_tflite": prob_tfl})
res.to_csv("/kaggle/working/holdout_predictions.csv", index=False)
info = {"n_total": len(df), "n_holdout": len(res), "seed": SEED, "tensorflow": tf.__version__,
        "h5": os.path.basename(h5_path), "tflite": os.path.basename(tflite_path),
        "max_abs_h5_tflite_diff": float(np.abs(prob_h5 - prob_tfl).max()),
        "h5_metrics_at_0.5": {k: float(v) for k, v in m_h5.items()},
        "reproducibility_problems": problems}
json.dump(info, open("/kaggle/working/holdout_run_info.json", "w"), indent=2)

print("\n" + "=" * 60)
if problems:
    print("REPRODUCIBILITY CHECK FAILED. Treat metrics as UNVERIFIED and tell Claude Code:")
    for p in problems: print(" -", p)
else:
    print("REPRODUCIBILITY CHECK PASSED: fold size, class counts, row order and h5 metrics all match training.")
print("Saved: /kaggle/working/holdout_predictions.csv and holdout_run_info.json")
